# DT-SPM — reproduce all manuscript results in one notebook

Runs on **Google Colab** (paste the Google Drive file id of `DT-SPM_data.zip` in the setup cell) or locally (keep `DT-SPM_data/` next to this notebook). Run top to bottom. Each section reproduces the numbers and figures of the manuscript from the saved run artifacts (`DT-SPM_data/output/**.npz`) and re-fits the learned models.

**Pipeline.** Measured force–distance (FD) curves and a scan archive → (1) descriptor extraction → (2) a physics-informed **PINN encoder** that recovers the descriptors → (3a) a deterministic **feedback scanner** re-parameterised by those descriptors → (3b) **CNN-LSTM corrections** that predict the scan-quality descriptors against experiment.

**What re-runs here.** The three learned models (PINN encoder, Step-3b Q_align correction, force-based Q_safety) re-fit in this notebook. The deterministic physics (RK45 FD fit; numba feedback scanner) is heavy and lives in the framework notebooks `DT-SPM_full_digital_twin.ipynb`; its outputs are loaded from `.npz`. The saved arrays are the canonical reference and reproduce the figures exactly.

Set `RETRAIN_PINN` / `RETRAIN_S3B` below to re-fit those two; the force-Q_safety model always re-fits (≈30 s).

In [ ]:
# ================== SETUP — Google Colab & local ==================
# Google Colab:
#   1. upload `DT-SPM_data.zip` to your Google Drive,
#   2. right-click it > Share > General access: 'Anyone with the link',
#   3. copy the share link and paste the file id below (the part between
#      /d/ and /view in the link),
#   4. run this cell — it downloads and unpacks the data package.
# Local run: unzip `DT-SPM_data.zip` (or keep the DT-SPM_data/ folder)
# next to this notebook and just run; the id can stay empty.

GDRIVE_FILE_ID = ''   # <-- paste the Google Drive file id of DT-SPM_data.zip here

import sys, subprocess, importlib.util, zipfile
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules

# install any missing packages (all of these ship preinstalled on Colab)
for mod, pip_name in [('torch', 'torch'), ('sklearn', 'scikit-learn'), ('gdown', 'gdown')]:
    if importlib.util.find_spec(mod) is None:
        print('installing', pip_name)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=False)

if IN_COLAB:
    DATA_ROOT = Path('/content/DT-SPM_data')
    if not (DATA_ROOT/'output').exists():
        assert GDRIVE_FILE_ID, ('GDRIVE_FILE_ID is empty — paste the Google Drive '
                                'file id of DT-SPM_data.zip above and re-run')
        import gdown
        zpath = '/content/DT-SPM_data.zip'
        gdown.download(id=GDRIVE_FILE_ID, output=zpath, quiet=False)
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall('/content')
        assert (DATA_ROOT/'output').exists(), 'unexpected zip layout — expected DT-SPM_data/output/'
else:
    _cands = [Path.cwd()/'DT-SPM_data', Path.cwd().parent/'DT-SPM_data', Path.cwd()]
    DATA_ROOT = next((p for p in _cands if (p/'output').exists()), None)
    assert DATA_ROOT is not None, ('DT-SPM_data/ not found — unzip DT-SPM_data.zip '
                                   'next to this notebook')

(DATA_ROOT/'output'/'pub_figures').mkdir(parents=True, exist_ok=True)
print('data package:', DATA_ROOT)

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')
%matplotlib inline

# ROOT = the data package root, resolved by the SETUP cell above
ROOT = DATA_ROOT
assert (ROOT/'output').exists(), f'data package missing (ROOT={ROOT}) — run the SETUP cell first'
sys.path.insert(0, str(ROOT/'codes'))
FIG_M = ROOT/'output'/'descriptor_framework'           # Multi-75 / calibration grating
FIG_T = ROOT/'output'/'descriptor_framework_Tap300'    # Tap-300 / AlScN
BUND_M = ROOT/'output'/'calibration_grating_v2_figures'/'figure_45_data_bundle.npz'
BUND_T = ROOT/'output'/'dt_controller_fit_figures'/'figure_45_expscan_data_bundle.npz'
import torch, torch.nn as nn
torch.manual_seed(0); np.random.seed(0)
RETRAIN_PINN = False     # True -> re-fit a compact Step-2 encoder (≈1 min)
RETRAIN_S3B  = False     # True -> re-fit a compact Step-3b Q_align correction (≈30 s)
print('ready · torch', torch.__version__)

## 0 · Load the saved artifacts
Every downstream result is computed from these. Shapes are printed so the data flow is explicit.

In [ ]:
def show(path, keys=None):
    z = np.load(path, allow_pickle=True)
    ks = keys or list(z.keys())
    print(f'  {Path(path).name}')
    for k in ks:
        a = z[k]
        try: print(f'     {k:20s} {str(a.shape):16s} {a.dtype}')
        except Exception: print(f'     {k:20s} scalar')
    return z

print('FORCE-DISTANCE libraries (frozen physics prior):')
fd_T = show(ROOT/'output'/'Tap300_AlScN.npz', ['amp','phase','drive'])
fd_M = show(ROOT/'output'/'cali_fd.npz', ['amp','phase','drive'])
print('\nStep-2 PINN descriptor predictions (per FD curve):')
pp_T = show(FIG_T/'pinn_step2_predictions.npz', ['anchors_z','mu','sigma','fd_train_idx','fd_test_idx','vocab_cols'])
pp_M = show(FIG_M/'pinn_step2_predictions.npz', ['anchors_z','mu','sigma','fd_train_idx','fd_test_idx'])
print('\nStep-3a scanner traces + Step-3b correction outputs + scan archive bundle:')
sc_T = show(FIG_T/'scanner_PINN_traces.npz', ['dt_PINN'])
s3b_T = show(FIG_T/'step3b_corrected_Q.npz', ['Q_scanner','Q_pred','Q_exp','is_test'])
bnd_T = show(BUND_T, ['traces_exp','dt_PI','dd','hyb','q_exp','q_PI','q_dd','q_hyb','rmse_PI','rmse_dd','rmse_hyb','test_idx','drive','setpoint','igain','scan_speed'])
print('\nAmplitude/phase scan lines (for force-based Q_safety) + phase-decoupling fields:')
jc_T = show(FIG_T/'joint_channel_calibration.npz', ['A_exp','PHI_exp'])
mef_T = show(FIG_T/'model_error_fields.npz', ['sin_target','sin_null','sin_pred','curve','fd_test_idx'])

## 1 · Step 1 — descriptor vocabulary
Each FD curve is reduced to the locked 18-descriptor vocabulary; each scan condition to the scan-quality descriptors. Tier weights encode physical importance (d_AR = 2.0).

In [ ]:
TW = {'A0':1,'phi_baseline':1,'Q_eff':1,'d_AR':2,'A_AR':1,'phi_att':1.5,'phi_rep':1.5,
      'phi_max':1.5,'d_phimax':.7,'d_50':.7,'A_50':.7,'d_70':.7,'A_70':.7,
      'slope_AR':.3,'slope_contact':.3,'d_contact':.7,'slope_50':.3,'slope_70':.3}
TIER1 = ['A0','phi_baseline','Q_eff','d_AR','A_AR','phi_att','phi_rep','phi_max',
         'd_phimax','d_50','A_50','d_70','A_70']
TIER2 = ['slope_AR','slope_contact','d_contact','slope_50','slope_70']
voc_T = pd.read_csv(FIG_T/'fd_vocab_v0_1.csv'); voc_M = pd.read_csv(FIG_M/'fd_vocab_v0_1.csv')
print('Tap-300 vocabulary (first 3 of 30 curves):')
display(voc_T[['A0','d_AR','phi_max','d_50','d_70','drive_nm']].head(3).round(2))

## 2 · Step 2 — physics-informed descriptor encoder
Held-out descriptor recovery from the **saved** encoder predictions — these are the canonical artifacts and reproduce the manuscript exactly (tier-weighted MAE 0.172 / 0.187). The next cell is an optional compact re-fit (`RETRAIN_PINN`); with 22/11 training curves it lands in the same regime but does not reproduce the tuned model to the decimal — the saved predictions above are the reference.

In [ ]:
VOCAB = [str(v) for v in pp_T['vocab_cols']]
def wagg(e, cols):
    cc = [c for c in cols if np.isfinite(e[c])]
    return float(np.average([e[c] for c in cc], weights=[TW[c] for c in cc]))
def step2_metrics(figdir):
    pp = np.load(figdir/'pinn_step2_predictions.npz')
    tgt = pd.read_csv(figdir/'fd_vocab_v0_1.csv')[VOCAB].to_numpy(float)
    tz = (tgt - pp['mu'])/pp['sigma']; pz = pp['anchors_z']
    tr, te = pp['fd_train_idx'], pp['fd_test_idx']
    def per_desc(P): return {c: np.nanmedian(np.abs((P-tz)[te, j])[np.isfinite(tgt[te, j])]) for j,c in enumerate(VOCAB)}
    prior = np.tile(np.nanmedian(tz[tr], 0, keepdims=True), (len(tz), 1))   # naive train-median prior
    eb, el = per_desc(prior), per_desc(pz)
    j = VOCAB.index('d_AR'); dAR = np.nanmedian(np.abs((pz-tz)[te, j]*pp['sigma'][j])[np.isfinite(tgt[te, j])])
    return wagg(eb, VOCAB), wagg(el, VOCAB), wagg(el, TIER1), wagg(el, TIER2), dAR

for name, figdir in [('Multi-75', FIG_M), ('Tap-300', FIG_T)]:
    base, a, t1, t2, dAR = step2_metrics(figdir)
    print(f'{name}: encoder held-out tier-weighted MAE(z) = {a:.3f}  vs train-median prior {base:.3f} '
          f'({100*(a-base)/base:+.0f}%) · Tier1 {t1:.3f} · Tier2 {t2:.3f} · d_AR median|err| ≈ {dAR:.2f} nm')

In [ ]:
# ---- optional re-fit of the Step-2 encoder (faithful compact version) ----
L = 128
def resample(curve, n=L):
    x = np.linspace(0, 1, len(curve)); return np.interp(np.linspace(0,1,n), x, curve)

class FDEncoder(nn.Module):
    def __init__(self, n_anchor=18, n_tok=4, hidden=64):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(2,16,7,padding=3), nn.GELU(),
            nn.Conv1d(16,32,5,padding=2), nn.GELU(), nn.AdaptiveAvgPool1d(8))
        self.mlp = nn.Sequential(nn.Linear(32*8+1+n_tok, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU())
        self.head = nn.Linear(hidden, n_anchor)
    def forward(self, A, phi, drive, tok):
        f = self.conv(torch.stack([A, phi], 1)).reshape(A.shape[0], -1)
        return self.head(self.mlp(torch.cat([f, drive[:,None], tok], -1)))

def build_inputs(fd, voc, pp, probe_oh):
    A0 = voc['A0'].to_numpy(float)
    A = np.stack([resample(fd['amp'][i]/A0[i]) for i in range(len(A0))]).astype(np.float32)
    phi = np.stack([resample(fd['phase'][i])/180.0 for i in range(len(A0))]).astype(np.float32)
    tgt = voc[VOCAB].to_numpy(float); tz = np.nan_to_num((tgt-pp['mu'])/pp['sigma']).astype(np.float32)
    mask = np.isfinite(tgt).astype(np.float32)
    drive = (voc['drive_nm'].to_numpy(float)/voc['drive_nm'].max()).astype(np.float32)
    tok = np.tile(probe_oh, (len(A0),1)).astype(np.float32)
    return A, phi, drive, tok, tz, mask

tw = np.array([TW[c] for c in VOCAB], np.float32)
if RETRAIN_PINN:
    for name, fd, voc, figdir, oh in [('Multi-75', fd_M, voc_M, FIG_M, [1,0,1,0]),
                                      ('Tap-300', fd_T, voc_T, FIG_T, [0,1,0,1])]:
        pp = np.load(figdir/'pinn_step2_predictions.npz')
        A, phi, drive, tok, tz, mask = build_inputs(fd, voc, pp, oh)
        tr, te = pp['fd_train_idx'], pp['fd_test_idx']
        At, pt, dt, kt, yt, mt = (torch.tensor(x) for x in (A,phi,drive,tok,tz,mask))
        torch.manual_seed(0); m = FDEncoder(); opt = torch.optim.Adam(m.parameters(), 3e-3, weight_decay=1e-4)
        for ep in range(1500):
            m.train(); pr = m(At[tr], pt[tr], dt[tr], kt[tr])
            l = (torch.tensor(tw)*mt[tr]*(pr-yt[tr]).abs()).sum()/(mt[tr]*torch.tensor(tw)).sum()
            opt.zero_grad(); l.backward(); opt.step()
        m.eval()
        with torch.no_grad(): pz = m(At, pt, dt, kt).numpy()
        e = np.array([np.nanmedian(np.abs((pz-tz)[te, j])[mask[te, j]>0]) for j in range(18)])
        agg = np.average(e[np.isfinite(e)], weights=tw[np.isfinite(e)])
        print(f're-fit {name}: tier-weighted held-out MAE(z) ≈ {agg:.3f}')
else:
    print('RETRAIN_PINN=False — using the saved predictions above.')

## 3a · Scanner wiring and the fidelity mechanism
The encoder anchors re-parameterise the deterministic scanner (traces loaded). The AlScN fidelity gap is diagnosed: the controls carry the signal, but the deterministic twin and a gain re-simulation miss it; the operating-point amplification factor explains it.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
b = bnd_T; rz = np.load(FIG_T/'resim_physical.npz')
te = np.zeros(900, bool); te[b['test_idx']] = True
Qexp, Qpi, QA = rz['Q_exp'][:,0], rz['Q_pi'][:,0], rz['Q_A'][:,0]
X = np.column_stack([b['drive'], b['setpoint'], b['igain'], b['scan_speed']])
Xz = (X-X.mean(0))/X.std(0)
knn = KNeighborsRegressor(5).fit(Xz[~te], np.log1p(np.clip(Qexp[~te],0,None)))
pk = knn.predict(Xz[te]); m = np.isfinite(Qexp[te])
print('Q1  experimental scan quality is set-point-driven; the deterministic twin is flat.')
print(f'Q2  control-only kNN predicts held-out quality:    Spearman {stats.spearmanr(pk[m], np.clip(Qexp[te],0,None)[m]).statistic:+.2f}')
print(f'Q3a deterministic twin vs experiment:               Spearman {stats.spearmanr(Qpi[te], Qexp[te], nan_policy="omit").statistic:+.2f}')
print(f'Q3b global-gain re-simulation vs experiment:        Spearman {stats.spearmanr(QA[te], Qexp[te], nan_policy="omit").statistic:+.2f}')

# operating-point amplification factor 1/|dA/dd|(d_op), zero-fit from the FD library
fd_amp, fd_h = fd_T['amp'], np.load(ROOT/'output'/'Tap300_AlScN.npz')['height']
fd_drive = np.array([np.nanmean(fd_amp[i,-10:]) for i in range(fd_amp.shape[0])])
def amp_factor(V, s):
    i = int(np.argmin(np.abs(fd_drive-V))); d, A = fd_h[i], fd_amp[i]
    o = np.argsort(d); d, A = d[o], A[o]; mk = np.isfinite(d)&np.isfinite(A); d, A = d[mk], A[mk]
    A0 = np.nanmean(A[-10:]); At = s*A0; c = np.where(np.diff(np.sign(A-At))!=0)[0]
    if not len(c): return np.nan
    k = c[0]; k1,k2 = max(k-4,0), min(k+4,len(d)-1)
    return 1.0/max(abs((A[k2]-A[k1])/(d[k2]-d[k1]+1e-12)), 1e-4)
af = np.array([amp_factor(b['drive'][i], b['setpoint'][i]) for i in range(900)])
mm = te & np.isfinite(af) & (Qexp>0)
print(f'Q4  zero-fit amplification 1/|dA/dd|(d_op) vs quality: Spearman {stats.spearmanr(af[mm], Qexp[mm]).statistic:+.2f}')

## 3b · Step-3b correction — height quality Q_align
The CNN-LSTM corrects the scanner's Q_align against experiment. Held-out metrics from the **saved** correction are canonical: on Tap-300/AlScN the median scan-quality error drops 10.4 → 0.6 nm and Spearman rises to +0.67. The following cell is an optional compact re-fit (`RETRAIN_S3B`) that illustrates the CNN-LSTM training pipeline.

In [ ]:
z = s3b_T; te2 = z['is_test']
for name, zz in [('Tap-300', s3b_T), ('Multi-75', np.load(FIG_M/'step3b_corrected_Q.npz'))]:
    t = zz['is_test']; Qe, Qs, Qp = zz['Q_exp'][:,0], zz['Q_scanner'][:,0], zz['Q_pred'][:,0]
    pm = np.nanmedian(np.abs(Qs-Qe)[t]); cm = np.nanmedian(np.abs(Qp-Qe)[t])
    rp = stats.spearmanr(Qe[t], Qs[t]).statistic; rc = stats.spearmanr(Qe[t], Qp[t]).statistic
    print(f'{name} Q_align: median |err| {pm:.2f} -> {cm:.2f} nm · Spearman {rp:+.2f} -> {rc:+.2f}')

In [ ]:
# ---- optional re-fit of the Step-3b Q_align correction (CNN-LSTM over each scan line) ----
# Operates on the 869 saved windows; win_cond maps each to its scan condition.
if RETRAIN_S3B:
    wc = s3b_T['win_cond']                       # (869,) condition index per saved window
    Qe = s3b_T['Q_exp'][:,0]; Qpr = s3b_T['Q_scanner'][:,0]; t = s3b_T['is_test']
    exp = b['traces_exp'][:,0]; sim = np.load(FIG_T/'scanner_PINN_traces.npz')['dt_PINN'][:,0]
    def ctr(a): return a - np.nanmedian(a[...,8:-8], -1, keepdims=True)
    lines = np.nan_to_num(np.stack([ctr(exp[wc]), ctr(sim[wc])], 1)).astype(np.float32)  # (869,2,256)
    med = np.median((Qe-Qpr)[~t]); iqr = np.subtract(*np.percentile((Qe-Qpr)[~t],[75,25]))+1e-6
    y = ((Qe-Qpr)-med)/iqr
    class Corr(nn.Module):                       # CNN local features -> LSTM over the line -> ΔQ
        def __init__(s, nf=24, h=48):
            super().__init__(); s.cnn = nn.Sequential(nn.Conv1d(2,16,9,2,4), nn.GELU(),
                nn.Conv1d(16,32,9,2,4), nn.GELU(), nn.Conv1d(32,nf,9,2,4), nn.GELU())  # (B,nf,32)
            s.lstm = nn.LSTM(nf, h, batch_first=True); s.head = nn.Sequential(nn.Linear(h,24), nn.GELU(), nn.Linear(24,1))
        def forward(s, L):
            f = s.cnn(L).transpose(1,2); o,_ = s.lstm(f); return s.head(o[:,-1]).squeeze(-1)
    lt = torch.tensor(lines); yt = torch.tensor(y.astype(np.float32)); tr = np.where(~t)[0]
    torch.manual_seed(0); m = Corr(); opt = torch.optim.Adam(m.parameters(), 2e-3, weight_decay=1e-4); hub = nn.HuberLoss(delta=1.0)
    for ep in range(300):
        bi = np.random.choice(tr, min(128,len(tr)), False)
        opt.zero_grad(); hub(m(lt[bi]), yt[bi]).backward(); opt.step()
    m.eval()
    with torch.no_grad(): Qp = Qpr + m(lt).numpy()*iqr+med
    print(f're-fit Tap-300 Q_align: held-out Spearman {stats.spearmanr(Qe[t], Qp[t]).statistic:+.2f} '
          f'(prior {stats.spearmanr(Qe[t], Qpr[t]).statistic:+.2f}; median|err| {np.nanmedian(np.abs(Qp-Qe)[t]):.2f} nm)')
else:
    print('RETRAIN_S3B=False — using the saved correction above.')

## 3b · Force-based Q_safety from amplitude + phase (full re-fit)
Q_safety is read from the amplitude and phase traces, not the height. `force = (A0−A)/A0 + ½·max(0,(90°−φ)/90°)`, 90th percentile along the line. The deterministic prior is the steady force from the set-point and the FD phase; a CNN-LSTM over the amplitude/phase window corrects it. This cell re-fits the model.

In [ ]:
Ae, Pe = jc_T['A_exp'], jc_T['PHI_exp']
sde = pd.read_csv(FIG_T/'scan_desc_pinn.csv'); ex = sde[sde.source=='exp'].reset_index(drop=True)
sp, A0c, ig = ex.setpoint.values.astype(float), ex.drive.values.astype(float), ex.igain.values.astype(float)
is_test = ex.is_test.values.astype(bool); n = len(ex)
def wrap(p): p = np.mod(p, 360.0); return np.where(p>180, p-360, p)
def force_line(A, phi, a0): 
    mk = np.isfinite(A)&np.isfinite(phi)
    return (a0-A[mk])/a0 + 0.5*np.maximum(0,(90-wrap(phi[mk]))/90)
Q_exp = np.array([np.nanmean([np.nanpercentile(force_line(Ae[i,d],Pe[i,d],A0c[i]),90) for d in (0,1)]) for i in range(n)])
# deterministic prior: steady force from set-point + FD phase at the operating point
fdp = np.load(ROOT/'output'/'Tap300_AlScN.npz')['phase']
def phi_op(V, s):
    i = int(np.argmin(np.abs(fd_drive-V))); A = fd_amp[i]; At = s*np.nanmean(A[-10:])
    c = np.where(np.diff(np.sign(A-At))!=0)[0]
    return float(fdp[i][c[0]]) if len(c) else np.nan
po = np.array([phi_op(A0c[i], sp[i]) for i in range(n)]); po = np.where(np.isfinite(po), po, np.nanmedian(po))
Q_prior = (1-sp) + 0.5*np.maximum(0,(90-wrap(po))/90)
rho = lambda a,bb,mk: float(stats.spearmanr(a[mk], bb[mk]).statistic)
print(f'prior held-out: Spearman {rho(Q_prior,Q_exp,is_test):+.3f}  median|e| {np.nanmedian(np.abs(Q_prior-Q_exp)[is_test]):.3f}')

In [ ]:
# CNN-LSTM correction over the (amplitude-reduction, repulsive-phase) window
af = np.array([amp_factor(A0c[i], sp[i]) for i in range(n)])  # operating-point feature
af = np.where(np.isfinite(af), af, np.nanmedian(af))
def cond_line(i):
    out = [np.stack([np.nan_to_num((A0c[i]-Ae[i,d])/A0c[i]),
                     np.nan_to_num(np.clip((90-wrap(Pe[i,d]))/90,-1,1))]) for d in (0,1)]
    return np.nanmean(out, 0)
lines = np.stack([cond_line(i) for i in range(n)]).astype(np.float32)
feat = np.stack([np.log10(af), sp, A0c/A0c.max(), ig/ig.max()], 1).astype(np.float32)
tr = ~is_test; med = np.median((Q_exp-Q_prior)[tr]); iqr = np.subtract(*np.percentile((Q_exp-Q_prior)[tr],[75,25]))+1e-6
y = ((Q_exp-Q_prior)-med)/iqr
K = 32; wc = np.arange(K-1, n)
widx = np.stack([[max(0,c-K+1+j) for j in range(K)] for c in wc])
for r,c in enumerate(wc):
    idx = list(range(max(0,c-K+1), c+1)); widx[r] = ([idx[0]]*(K-len(idx))+idx)[-K:]
class QS(nn.Module):
    def __init__(s, nf=24, h=48):
        super().__init__(); s.cnn = nn.Sequential(nn.Conv1d(2,16,9,2,4), nn.GELU(),
            nn.Conv1d(16,32,9,2,4), nn.GELU(), nn.Conv1d(32,nf,9,2,4), nn.GELU(), nn.AdaptiveAvgPool1d(1))
        s.lstm = nn.LSTM(nf+feat.shape[1], h, batch_first=True); s.head = nn.Sequential(nn.Linear(h,24), nn.GELU(), nn.Linear(24,1))
    def enc(s, L): return s.cnn(L).squeeze(-1)
    def fwd(s, e, f, wi): o,_ = s.lstm(torch.cat([e[wi], f[wi]], -1)); return s.head(o[:,-1]).squeeze(-1)
lt, ft, wt = torch.tensor(lines), torch.tensor(feat), torch.tensor(widx, dtype=torch.long)
yt = torch.tensor(y[wc].astype(np.float32)); trr = np.where(~is_test[wc])[0]
torch.manual_seed(0); m = QS(); opt = torch.optim.Adam(m.parameters(), 2e-3, weight_decay=1e-4); hub = nn.HuberLoss(delta=1.0)
for ep in range(300):
    e = m.enc(lt); bi = np.random.choice(trr, min(96,len(trr)), False)
    opt.zero_grad(); hub(m.fwd(e, ft, wt[bi]), yt[bi]).backward(); opt.step()
m.eval()
with torch.no_grad(): dq = m.fwd(m.enc(lt), ft, wt).numpy()*iqr+med
Q_pred = Q_prior.copy(); Q_pred[wc] += dq; tw_ = is_test[wc]
print(f'corrected held-out: Spearman {stats.spearmanr(Q_exp[wc][tw_], Q_pred[wc][tw_]).statistic:+.3f}  '
      f'median|e| {np.nanmedian(np.abs(Q_pred-Q_exp)[is_test]):.3f}')
ra = stats.spearmanr([np.nanmean([np.nanpercentile((A0c[i]-Ae[i,d])/A0c[i],90) for d in (0,1)]) for i in np.where(is_test)[0]], sp[is_test]).statistic
print(f'channel check: amplitude vs set-point Spearman {ra:+.2f} (force); phase is set-point-independent (branch)')
qc = np.load(FIG_T/'qsafety_ap.npz'); tc = qc['is_test']
print(f"canonical (qsafety_ap.npz): prior {stats.spearmanr(qc['Q_exp'][tc], qc['Q_prior'][tc]).statistic:+.3f}"
      f" -> corrected {stats.spearmanr(qc['Q_exp'][tc], qc['Q_pred'][tc]).statistic:+.3f}  (manuscript force Q_safety: +0.92 -> +0.97)")

## 4 · Physics vs data-driven vs hybrid (Multi-75 grating)
Three predictors of the same held-out scan lines on the manuscript Fig 7 system: a scalar scan-quality error and a line-shape RMSE. Physics wins the scalar quality, learning wins the line shape — the Pareto trade-off the hybrid is built to span.

In [ ]:
bM = np.load(BUND_M)                       # Multi-75 grating bundle (the manuscript Fig 7 system)
ti = bM['test_idx']; qe = bM['q_exp']
print('scalar scan-quality error (held-out MAE, nm)   — physics wins:')
for lab,k in [('pure physics','q_PI'),('pure data-driven','q_dd'),('hybrid','q_hyb')]:
    print(f'   {lab:18s} {np.nanmean(np.abs(bM[k][ti]-qe[ti])):.1f}')
print('line-shape error (held-out median RMSE, nm)    — learning wins:')
for lab,k in [('pure physics','rmse_PI'),('pure data-driven','rmse_dd'),('hybrid','rmse_hyb')]:
    print(f'   {lab:18s} {np.nanmedian(np.nanmean(bM[k][ti],1)):.0f}')

## 5 · Supplementary — phase channel decoupling
Amplitude is trusted as measured; the phase model error is one learned dissipation field via the energy-balance identity. Held-out median phase error vs the conservative null.

In [ ]:
for name, figdir in [('Multi-75', FIG_M), ('Tap-300', FIG_T)]:
    z = np.load(figdir/'model_error_fields.npz'); te4 = np.isin(z['curve'], z['fd_test_idx'])
    eb = np.nanmedian(np.abs(z['sin_null']-z['sin_target'])[te4])
    em = np.nanmedian(np.abs(np.clip(z['sin_pred'],-1,1)-z['sin_target'])[te4])
    print(f'{name}: held-out median |sin φ error|  null {eb:.4f} -> model {em:.4f}  ({100*(em-eb)/eb:+.0f}%)')

## 6 · Regenerate every manuscript figure
The figure code lives in `DT-SPM_data/codes/_make_*.py` . Importing the modules regenerates the PNGs in `DT-SPM_data/output/pub_figures/`; they are displayed below.

In [ ]:
from IPython.display import Image, display
import importlib
import _make_pro_figures as fpro
import _make_manuscript_figures_v2 as fv2
import _make_comparison_figures as fcmp
fpro.make_fig1_pro(); fpro.make_fig2_pro()
fv2.make_fig2(); fv2.make_fig3(); fv2.make_fig4(); fv2.make_fig5(); fv2.make_fig6()
fcmp.make_c1(); fcmp.make_c2(); fcmp.make_c3(); fcmp.make_c4()
import _make_pub_figures   # runs r1..r4 on import
PUB = ROOT/'output'/'pub_figures'
order = [('Fig 1', 'fig1_pro'), ('Fig 2', 'fig2_pro'), ('Fig 3', 'fig2_descriptors_Q'),
         ('Fig 4', 'fig3_step2_fit'), ('Fig 5', 'fig4_mechanism_Q'), ('Fig 6', 'fig5_step3b_predict'),
         ('Fig 7', 'fig6_three_models'), ('ED 1', 'c1_step2_fdfits'), ('ED 2', 'r1_step2_pinn'),
         ('ED 3', 'c2_step3a_traces'), ('ED 4', 'c3_step3b_correction'), ('ED 5', 'r3_correction'),
         ('Supp 1', 'c4_phase_decoupling'), ('Supp 2', 'r4_decoupling')]
for tag, stub in order:
    print(tag); display(Image(filename=str(PUB/f'{stub}.png')))

---
*Reproduction complete.* The learned models re-fit here; all metrics and figures match the manuscript. Deterministic-physics source: the framework notebooks `DT-SPM_full_digital_twin.ipynb`; learned-model source: `DT-SPM_data/codes/_qsafety_ap.py` and the patch scripts.